# KOSIS 뉴스 사실검증 PoC ERD

`kosis_schema_design.ipynb`에서 구현한 SQLite 스키마를 기준으로 작성한 ERD다.

```mermaid
erDiagram
    KOSIS_TABLES ||--o{ KOSIS_DIMENSIONS : has
    KOSIS_DIMENSIONS ||--o{ KOSIS_DIMENSION_VALUES : contains
    KOSIS_TABLES ||--o{ KOSIS_ITEMS : measures
    KOSIS_TABLES ||--o{ KOSIS_PERIODS : provides
    CLAIMS ||--o{ VERIFICATION_RESULTS : produces
    KOSIS_ITEMS ||--o{ VERIFICATION_RESULTS : supports

    KOSIS_TABLES {
        TEXT table_key PK "org_id:tbl_id"
        TEXT org_id UK "기관 코드"
        TEXT tbl_id UK "통계표 ID"
        TEXT tbl_name "통계표명"
        TEXT tbl_name_eng
        TEXT stat_id
        TEXT stat_name
        TEXT view_code
        TEXT category_path
        INTEGER recommended
        TEXT source_updated_at
        TEXT retrieved_at
    }

    KOSIS_DIMENSIONS {
        TEXT table_key PK,FK
        TEXT dimension_id PK
        TEXT dimension_name
        TEXT dimension_name_eng
        INTEGER dimension_order
        TEXT dimension_type
        INTEGER required
    }

    KOSIS_DIMENSION_VALUES {
        TEXT table_key PK,FK
        TEXT dimension_id PK,FK
        TEXT value_id PK
        TEXT value_name
        TEXT value_name_eng
        TEXT parent_value_id
        TEXT normalized_name
        TEXT aliases_json
    }

    KOSIS_ITEMS {
        TEXT table_key PK,FK
        TEXT item_id PK
        TEXT item_name
        TEXT item_name_eng
        TEXT unit_id
        TEXT unit_name
        TEXT metric_concept
        TEXT measure_type
        TEXT normalized_unit
        TEXT aliases_json
        TEXT claim_template
    }

    KOSIS_PERIODS {
        TEXT table_key PK,FK
        TEXT period_type PK
        TEXT period_name
        TEXT start_period
        TEXT end_period
        TEXT period_format
        TEXT reference_type
        INTEGER publication_lag_days
    }

    CLAIMS {
        TEXT claim_id PK
        TEXT article_id
        TEXT claim_text
        TEXT metric
        REAL claim_value
        TEXT claim_unit
        REAL normalized_value
        TEXT normalized_unit
        TEXT time_expression
        TEXT resolved_period
        TEXT period_type
        TEXT region
        TEXT population
        TEXT comparison_type
        TEXT verifiability
        TEXT extracted_at
        TEXT extraction_json
    }

    VERIFICATION_RESULTS {
        TEXT verification_id PK
        TEXT claim_id FK
        TEXT table_key FK
        TEXT item_id FK
        REAL official_value
        TEXT official_unit
        TEXT official_period
        TEXT dimensions_json
        REAL absolute_difference
        REAL relative_difference
        TEXT verdict
        TEXT reason_code
        TEXT explanation
        TEXT evidence_json
        TEXT review_status
        TEXT verified_at
    }

    RAW_API_RESPONSES {
        TEXT response_id PK
        TEXT endpoint
        TEXT request_params_json
        TEXT response_json
        TEXT retrieved_at
    }
```

## 데이터 연결 흐름

```mermaid
flowchart LR
    A[뉴스 기사] --> B[CLAIMS<br/>구조화된 수치 주장]
    B --> C[VERIFICATION_RESULTS<br/>비교 및 판정]

    D[KOSIS_TABLES<br/>통계표 카탈로그] --> E[KOSIS_ITEMS<br/>지표·항목·단위]
    D --> F[KOSIS_DIMENSIONS<br/>분류 축]
    F --> G[KOSIS_DIMENSION_VALUES<br/>지역·성별·연령 등]
    D --> H[KOSIS_PERIODS<br/>수록주기·시점 범위]

    E --> C
    G -. 조건 정렬 .-> C
    H -. 시점 정렬 .-> C
    I[RAW_API_RESPONSES<br/>원본 응답] -. 재현·감사 .-> D
```

## 관계 및 카디널리티

| 부모 | 관계 | 자식 | 의미 |
|---|---:|---|---|
| `kosis_tables` | 1:N | `kosis_dimensions` | 한 통계표는 여러 분류 축을 가질 수 있다. |
| `kosis_dimensions` | 1:N | `kosis_dimension_values` | 하나의 분류에는 여러 선택값이 있다. |
| `kosis_tables` | 1:N | `kosis_items` | 한 통계표는 여러 지표 항목을 측정할 수 있다. |
| `kosis_tables` | 1:N | `kosis_periods` | 한 통계표는 연·월 등 복수 수록주기를 가질 수 있다. |
| `claims` | 1:N | `verification_results` | 한 주장을 여러 후보·버전으로 검증할 수 있다. |
| `kosis_items` | 1:N | `verification_results` | 하나의 공식 지표가 여러 뉴스 주장에 근거로 사용될 수 있다. |

`raw_api_responses`는 KOSIS 정규화 테이블과 의도적으로 외래키를 연결하지 않았다. API 실패 응답이나 아직 정규화하지 않은 응답도 그대로 보존해야 하기 때문이다.

## 복합키와 외래키

- 통계표 자연키: `(org_id, tbl_id)`
- 내부 통계표 키: `table_key = org_id + ':' + tbl_id`
- 분류 PK: `(table_key, dimension_id)`
- 분류값 PK: `(table_key, dimension_id, value_id)`
- 항목 PK: `(table_key, item_id)`
- 수록주기 PK: `(table_key, period_type)`
- 검증 결과의 공식 근거 FK: `(table_key, item_id) → kosis_items`

뉴스 주장의 지역·모집단과 KOSIS 분류값 연결은 현재 `dimensions_json`에 저장한다. PoC 이후 매핑 이력과 후보 점수를 관리해야 할 때 `claim_dimension_mappings`와 `claim_table_candidates`를 추가한다.

In [1]:
# 실제 SQLite DB의 테이블 및 외래키가 ERD와 일치하는지 확인
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path.cwd() / 'output' / 'kosis_poc.db'
conn = sqlite3.connect(DB_PATH)
table_names = [row[0] for row in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()]

relationships = []
for table in table_names:
    for fk in conn.execute(f'PRAGMA foreign_key_list({table})').fetchall():
        relationships.append({
            'child_table': table, 'child_column': fk[3],
            'parent_table': fk[2], 'parent_column': fk[4],
            'on_delete': fk[6]
        })
pd.DataFrame(relationships)

OperationalError: unable to open database file